In [1]:
!pip install sentence-transformers scikit-learn pandas numpy

In [5]:
from google.colab import files
uploaded = files.upload()

Saving mechanic_logs_1000.csv to mechanic_logs_1000.csv


In [7]:
import pandas as pd

df = pd.read_csv("mechanic_logs_1000.csv")
logs = df["log"].tolist()

print("Total logs:", len(logs))

Total logs: 1000


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
embeddings = model.encode(logs, show_progress_bar=True)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [11]:
from sklearn.cluster import MiniBatchKMeans

num_clusters = 20

kmeans = MiniBatchKMeans(
    n_clusters=num_clusters,
    random_state=42,
    batch_size=256
)

labels = kmeans.fit_predict(embeddings)
df["cluster"] = labels

In [13]:
from collections import Counter

cluster_names = {}

for i in range(num_clusters):
    text = " ".join(df[df["cluster"] == i]["log"])
    words = text.split()

    common_words = Counter(words).most_common(3)
    name = " ".join([w[0] for w in common_words])

    cluster_names[i] = name

# preview
for i in range(20):
    print(i, ":", cluster_names[i])

0 : brake fluid leakage
1 : drains quickly overnight
2 : motor noise vibration
3 : charging interrupted randomly
4 : brake not responding
5 : vehicle shuts down
6 : wheel vibration above
7 : ac not cooling
8 : power window not
9 : overheating after long
10 : brake making noise
11 : headlight not working
12 : charging not issue
13 : jerk in during
14 : steering pulling to
15 : brake delay observed
16 : battery charging cable
17 : vibration during braking
18 : error code showing
19 : system reboot required


In [14]:
def generate_solution(cluster_name):
    return f"Inspect issues related to: {cluster_name}. Perform detailed diagnostics and maintenance."

In [15]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def detect_multiple_issues(text, top_k=3):
    emb = model.encode([text])
    sims = cosine_similarity(emb, kmeans.cluster_centers_)[0]

    top_indices = np.argsort(sims)[-top_k:][::-1]
    return top_indices, sims

In [16]:
def get_confidence(sims):
    return float(max(sims))

In [17]:
def generate_followup(cluster_list):
    questions = []

    for name in cluster_list:
        if "battery" in name:
            questions.append("Is the battery heating during charging or driving?")
        if "motor" in name:
            questions.append("Do you feel vibration or hear noise while driving?")
        if "charging" in name:
            questions.append("Are you facing slow or interrupted charging?")
        if "brake" in name:
            questions.append("Do you notice delay or noise while braking?")

    return list(set(questions))

In [18]:
def detect_severity(text):
    text = text.lower()

    high = ["failure", "not working", "shutdown", "leak"]
    medium = ["noise", "vibration", "overheating"]

    if any(word in text for word in high):
        return "HIGH"
    elif any(word in text for word in medium):
        return "MEDIUM"
    else:
        return "LOW"

In [19]:
def smart_mechanic_agent(user_input):

    indices, sims = detect_multiple_issues(user_input)

    cluster_list = [cluster_names[i] for i in indices]
    confidence = get_confidence(sims)
    severity = detect_severity(user_input)

    result = {}

    result["Detected Issues"] = cluster_list
    result["Confidence"] = round(confidence, 2)
    result["Severity"] = severity

    if confidence < 0.5:
        result["Action"] = "Need More Info"
        result["Follow-up Questions"] = generate_followup(cluster_list)
    else:
        result["Action"] = "Provide Solution"
        result["Solutions"] = [generate_solution(c) for c in cluster_list]

    return result

In [20]:
print("🚗 Industrial EV Mechanic AI (type 'exit' to stop)\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    result = smart_mechanic_agent(user_input)

    print("\n🤖 AI Response:")

    print("🔍 Issues Detected:", result["Detected Issues"])
    print("⚠️ Confidence:", result["Confidence"])
    print("📊 Severity:", result["Severity"])

    if result["Action"] == "Need More Info":
        print("\n❓ Follow-up Questions:")
        for q in result["Follow-up Questions"]:
            print("-", q)
    else:
        print("\n🛠️ Suggested Actions:")
        for sol in result["Solutions"]:
            print("-", sol)

    print("\n" + "="*50)

🚗 Industrial EV Mechanic AI (type 'exit' to stop)

You: battery overheating while fast charging

🤖 AI Response:
🔍 Issues Detected: ['battery charging cable', 'overheating after long', 'charging not issue']
⚠️ Confidence: 0.77
📊 Severity: MEDIUM

🛠️ Suggested Actions:
- Inspect issues related to: battery charging cable. Perform detailed diagnostics and maintenance.
- Inspect issues related to: overheating after long. Perform detailed diagnostics and maintenance.
- Inspect issues related to: charging not issue. Perform detailed diagnostics and maintenance.

You: battery draining fast and charging is slow

🤖 AI Response:
🔍 Issues Detected: ['charging not issue', 'drains quickly overnight', 'battery charging cable']
⚠️ Confidence: 0.71
📊 Severity: LOW

🛠️ Suggested Actions:
- Inspect issues related to: charging not issue. Perform detailed diagnostics and maintenance.
- Inspect issues related to: drains quickly overnight. Perform detailed diagnostics and maintenance.
- Inspect issues relate

KeyboardInterrupt: Interrupted by user